# 1-board: current-scale

Measure the current chain's scale and zero against the meter, so the firmware's counts per amp is a graded number and not a parts list.

## Purpose

The firmware limits, trips and identifies in amperes it computes from one ADC count through the shunt and its amplifier. The registry carries the parts' nominal values, which give a nominal counts per amp. This experiment grades that number against the meter on DC, where nothing but the chain can disagree, and finds the zero under drive, which the torque-off rest cannot show.

## Procedure

DC laps on a known load at 100 % duty, where the bridge does not switch and the chip and the meter read the same DC current. The DMM sits in series with the load as the ammeter. The DPS steps through levels in two bands, 4.5 to 5.5 V and 7.2 to 8 V, because the sweep tool's drive gate refuses a rail between them as a flat pack. Two loads give two current ranges: the bare 5 ohm up to 1.5 A and the series 15 ohm up to 0.5 A.

1. The board on the DPS, the adapter on the bus at 3 Mbps.
2. The DMM on DC A, red probe in the A jack, in series with the load across the motor terminals.
3. `capture.py bare`, then `capture.py series` with the other load. Each pass raises the servo's current limit for the laps and puts the saved one back after. Each level is one `osc sweep` of a one second rest and a three second lap in each direction.
4. Optional: `capture.py rs1` with the DMM on DC V across the shunt pads, the amplifier's own input, which grades the amplifier and the ADC with the shunt's value left out.

## Data

- `<pass>/levels.csv`: one row per level, the rail level asked for, the DPS voltage that reached it and the rail the board read at rest, and the host time the sweep began and ended.
- `<pass>/laps/<k>/sweep.csv`, `meta.json`: the rest and the two laps at that level, one row per control tick.
- `<pass>/dmm.csv`: the meter's log through the pass, `t_host`, `function`, `value`.

### levels.csv and the meter log

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oscnb import b41t, boards, display, laps, results, tel
from oscnb.display import show

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#898781"
BOARD = boards.BOARDS[boards.DEFAULT]
CAPTURE = Path("../../../telemetry") / BOARD.key / "current-scale" / "capture-1"
CAPTURE = CAPTURE.resolve()
print(BOARD.key, "from", CAPTURE)

In [ ]:
passes = [p.name for p in CAPTURE.iterdir() if (p / "levels.csv").exists()]
print("passes:", passes)
levels = pd.concat([pd.read_csv(CAPTURE / p / "levels.csv").assign(pass_=p) for p in passes], ignore_index=True)
show(levels, n=None)
dmm = {p: b41t.load(CAPTURE / p / "dmm.csv") for p in passes}
fig, ax = plt.subplots(1, len(passes), figsize=(6 * len(passes), 3.5), squeeze=False)
for a, p in zip(ax[0], passes):
    d = dmm[p]
    a.plot(d["t"], d["value"], ".", ms=2, color=BLUE); a.set_title(f"{p}: the meter"); a.set_xlabel("time (s)"); a.grid(alpha=0.3)
    for _, L in levels[levels["pass_"] == p].iterrows():
        a.axvspan(L["t_start"] - d["t_host"].iloc[0], L["t_end"] - d["t_host"].iloc[0], color=ORANGE, alpha=0.15)
plt.show()

The levels of each pass, as rail volts asked for and read, and the meter's log through it, with each level's sweep shaded. Inside a sweep the meter reads zero in the rest and the lap current in each direction.

### One lap

In [ ]:
p0 = passes[0]
df, meta = tel.load(CAPTURE / p0 / levels[levels["pass_"] == p0]["folder"].iloc[-1], BOARD)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(df["t"], df["current_raw"], lw=0.5, color=BLUE); ax[0].set_ylabel("current_raw (counts)"); ax[0].set_xlabel("time (s)"); ax[0].grid(alpha=0.3)
ax[1].plot(df["t"], df["vmotor_a"], lw=0.5, color=BLUE, label="tap A"); ax[1].plot(df["t"], df["vmotor_b"], lw=0.5, color=ORANGE, label="tap B")
ax[1].set_ylabel("counts"); ax[1].set_xlabel("time (s)"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.show()
print("gaps:", tel.gaps(df).to_dict() or "none")

The highest lap of the first pass: the rest, then the lap each way. The current channel reads the same current in both directions, since the shunt sits in the return of both; the taps swap.

## Method

Every lap gives one point: the chip's current counts over its own rest zero against the meter's amps in the same window. A straight line through all points of the ammeter passes is the chain: its slope is counts per amp, and its intercept is the offset the drive adds over the torque-off zero. The nominal counts per amp from the registry's parts is the comparison. The two directions are also fitted apart, since a difference between them belongs to the board's copper, not to the shunt.

The meter's reading in a lap is the median of its packets inside the sweep's window less a margin either side, which drops the rest and the switch-over. The rest zero is the median of the current channel over the second half of the segment 0 rest.

### The lap table

In [ ]:
tables = {}
for p in passes:
    t = laps.table(CAPTURE / p, BOARD, dmm[p])
    t["counts"] = t["current_raw"] - t["zero"]
    t["amps_chip"] = BOARD.amps(t["current_raw"], t["zero"])
    tables[p] = t
T = pd.concat([t.assign(pass_=p) for p, t in tables.items()], ignore_index=True)
show(T[["pass_", "k", "volts_set", "vsys_read", "dir", "zero", "counts", "amps_chip", "meter", "meter_n", "current_sd", "n"]], n=None)

One row per lap and direction: the rest zero, the counts over it, the chip's amps at the nominal scale, and the meter's reading with the number of packets it came from.

### The fit

In [ ]:
amm = T[T["pass_"].isin(["bare", "series"])].dropna(subset=["meter"])
fit = laps.line(amm["meter"].abs(), amm["counts"])
scale_err = (fit["slope"] / BOARD.counts_per_a - 1) * 100
print(f"{len(amm)} laps: {fit['slope']:.1f} +/- {fit['slope_se']:.1f} counts per amp against {BOARD.counts_per_a:.1f} nominal, {scale_err:+.2f} %; "
      f"offset under drive {fit['intercept']:+.1f} +/- {fit['intercept_se']:.1f} counts; residual rms {fit['rms']:.2f} counts")
by_dir = {d: laps.line(g["meter"].abs(), g["counts"]) for d, g in amm.groupby("dir") if len(g) >= 3}
for d, f in by_dir.items():
    print(f"  dir {d:+d}: {f['slope']:.1f} counts per amp, offset {f['intercept']:+.1f}")
direction_pct = (by_dir[-1]["slope"] / by_dir[1]["slope"] - 1) * 100 if len(by_dir) == 2 else np.nan
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for p, m in (("bare", "o"), ("series", "s")):
    g = amm[amm["pass_"] == p]
    ax[0].plot(g["meter"].abs(), g["counts"], m, color=BLUE, label=p)
x = np.linspace(0, amm["meter"].abs().max(), 10)
ax[0].plot(x, fit["slope"] * x + fit["intercept"], "-", color=INK, lw=0.8, label="fit"); ax[0].plot(x, BOARD.counts_per_a * x, "--", color=MUTED, lw=0.8, label="nominal")
ax[0].set_xlabel("meter (A)"); ax[0].set_ylabel("counts over the rest zero"); ax[0].legend(); ax[0].grid(alpha=0.3)
resid = amm["counts"] - (fit["slope"] * amm["meter"].abs() + fit["intercept"])
ax[1].plot(amm["meter"].abs(), resid / amm["counts"] * 100, "o", color=BLUE); ax[1].axhline(0, color=INK, lw=0.5)
ax[1].set_xlabel("meter (A)"); ax[1].set_ylabel("residual (% of the reading)"); ax[1].grid(alpha=0.3)
plt.show()

Left, counts against the meter's amps for every lap, the fitted line and the nominal one. Right, what is left after the fit, as a share of each reading.

### The shunt pads, when that pass was run

In [ ]:
gain = None
if "rs1" in tables:
    r = tables["rs1"].dropna(subset=["meter"])
    g = laps.line(r["meter"].abs() * 1e3, r["counts"])     # counts per mV across the shunt
    gain = g["slope"] * BOARD.v_adc_per_count * 1e3           # counts per mV x V per count = V/V gain
    print(f"amplifier gain {gain:.3f} against {BOARD.gain_milli / 1e3:.3f} nominal, {(gain / (BOARD.gain_milli / 1e3) - 1) * 100:+.2f} %; "
          f"offset {g['intercept']:+.1f} counts; shunt implied {fit['slope'] / g['slope'] * 1e-3 * 1e3:.2f} mOhm" if gain else "")
else:
    print("no rs1 pass; the amplifier and the shunt are not separated")

With the meter across the shunt pads, counts per millivolt is the amplifier's gain with the shunt left out, and the ammeter slope over it is the shunt's value.

## Results

The scale as counts per amp with its standard error as the bounds, its error against the nominal, the offset under drive, the direction difference, and the mean rest zero.

In [ ]:
results.write("1-board", "current-scale", CAPTURE, pair=BOARD.key,
              notes="DC laps at 100 % duty on the bare and series loads, DMM in series; counts over the torque-off rest zero against the meter",
              counts_per_a=results.quantity(fit["slope"], lo=fit["slope"] - fit["slope_se"], hi=fit["slope"] + fit["slope_se"], unit="counts/A"),
              scale_error_pct=results.quantity(scale_err, unit="%"),
              drive_offset_counts=results.quantity(fit["intercept"], lo=fit["intercept"] - fit["intercept_se"], hi=fit["intercept"] + fit["intercept_se"], unit="counts"),
              direction_pct=results.quantity(direction_pct, unit="%"),
              rest_zero_counts=results.quantity(T["zero"].mean(), lo=T["zero"].min(), hi=T["zero"].max(), unit="counts"),
              **({"amp_gain": results.quantity(gain, unit="V/V")} if gain else {}))

## Limitations

- The meter's DC A accuracy bounds everything here, and the 20 A jack reads a fixed 1.5 mA or so low under half an amp, as the power-resistors experiment found. The series load's points sit where that matters; the bare load's do not.
- The lap is DC. Under switching the crest sample lands on a rising current, and the sample-timing experiment is where that error lives.

## Open questions

- Whether the offset under drive is in the amplifier or in the bridge's return path. A lap with the meter across the shunt pads and one in series at the same level, back to back, separate them.

## Conclusion

To be written from the capture.